# Swahili NLLB fine-tune — 600M (full fine-tune, CORRECTED)

Fine-tunes **facebook/nllb-200-distilled-600M** on your clean parallel data.
Purpose-built translator — no CPT, no chat template, no 4-bit, no LoRA. One run.

**Fixes vs the first version:** loads weights in fp32 (lets `fp16=True` do the
mixed-precision casting correctly), and uses `adamw_torch` (no bitsandbytes needed).

Target: beat Crane's **27.6 BLEU / 56.8 chrF++** on FLORES en->sw — at *half*
Crane's parameters (the efficiency flex). Writes to `swa-nllb-600m` / `-merged`.

> **Setup:** GPU **T4 ×2** (uses one), Internet ON, `HF_TOKEN` secret, FLORES+ terms
> accepted. Restart kernel, then Run All. ~2h.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
# 600M full fine-tune needs NO bitsandbytes (adamw_torch) and NO torchao. Just sacrebleu.
!pip install -q sacrebleu bitsandbytes

In [ ]:
import torch, transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF auth OK | CUDA:", torch.cuda.is_available())

## Configuration

In [ ]:
HF_USERNAME  = "Dev-Noble"
MODEL_ID     = "facebook/nllb-200-distilled-600M"
HUB_CKPT     = f"{HF_USERNAME}/swa-nllb-600m"
HUB_MERGED   = f"{HF_USERNAME}/swa-nllb-600m-merged"

SRC_LANG, TGT_LANG = "eng_Latn", "swh_Latn"
MAX_LEN      = 128
LR           = 3e-5
NUM_EPOCHS   = 3
PER_DEV_BS   = 4
GRAD_ACCUM   = 8           # effective batch 32
WARMUP_RATIO = 0.03
SAVE_STEPS   = 200
LOG_STEPS    = 20
CAP = {"mafand": None, "salt": None, "rogendo": 60_000, "inkuba_mmt": 30_000}
OUTPUT_DIR = "/kaggle/working/nllb_out"; os.makedirs(OUTPUT_DIR, exist_ok=True)
assert HF_USERNAME != "YOUR_HF_USERNAME", "Set HF_USERNAME."
print("model:", MODEL_ID, "| effective batch:", PER_DEV_BS*GRAD_ACCUM)

## Tokenizer

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, src_lang=SRC_LANG)
print("vocab:", tok.vocab_size, "| tgt id:", tok.convert_tokens_to_ids(TGT_LANG))

## Load clean parallel pairs (no instruction data — NLLB is a pure translator)

In [ ]:
from datasets import load_dataset
pairs = []
def cap(name):
    n = CAP.get(name); return f"train[:{n}]" if n else "train"

try:
    _mf = "https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/json_files/en-swa"
    d = load_dataset("json", data_files={"tr": f"{_mf}/train.json", "dv": f"{_mf}/dev.json"}, split="tr+dv")
    b=len(pairs)
    for r in d:
        t=r["translation"]; en,sw=t.get("en"),t.get("swa")
        if en and sw and len(str(en))>5 and len(str(sw))>5: pairs.append((str(en),str(sw)))
    print("MAFAND:", len(pairs)-b)
except Exception as e: print("MAFAND FAILED:", e)

try:
    d = load_dataset("Rogendo/English-Swahili-Sentence-Pairs", split=cap("rogendo")); b=len(pairs)
    for r in d:
        en,sw=r.get("English sentence",""),r.get("Swahili Translation","")
        if len(str(en))>5 and len(str(sw))>5: pairs.append((str(en),str(sw)))
    print("Rogendo:", len(pairs)-b)
except Exception as e: print("Rogendo FAILED:", e)

try:
    d = load_dataset("Sunbird/salt", "text-all", split=cap("salt")); b=len(pairs)
    for r in d:
        en,sw=r.get("eng_source_text",""),r.get("swa_text","")
        if len(str(en))>5 and len(str(sw))>5: pairs.append((str(en),str(sw)))
    print("SALT:", len(pairs)-b)
except Exception as e: print("SALT FAILED:", e)

try:
    d = load_dataset("Svngoku/Inkuba-Swahili-MMT", split=cap("inkuba_mmt")); b=len(pairs)
    for r in d:
        en,sw=r.get("targets",""),r.get("inputs","")
        if len(str(en))>5 and len(str(sw))>5: pairs.append((str(en),str(sw)))
    print("Inkuba-MMT:", len(pairs)-b)
except Exception as e: print("Inkuba-MMT FAILED:", e)

import random
random.seed(42); random.shuffle(pairs)
print("TOTAL parallel pairs:", len(pairs))

## Tokenize (source -> target)

In [ ]:
from datasets import Dataset
raw = Dataset.from_dict({"en": [e for e,_ in pairs], "sw": [s for _,s in pairs]})
def tok_fn(batch):
    mi = tok(batch["en"], max_length=MAX_LEN, truncation=True)
    lbl = tok(text_target=batch["sw"], max_length=MAX_LEN, truncation=True)
    mi["labels"] = lbl["input_ids"]; return mi
tokd = raw.map(tok_fn, batched=True, remove_columns=["en","sw"], desc="tokenize")
print("tokenized examples:", len(tokd))

## Load model — fp32 (full fine-tune). `fp16=True` handles mixed precision.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","out_proj","fc1","fc2"], bias="none"))
model.print_trainable_parameters()

## Resume-from-Hub + train

In [ ]:
from huggingface_hub import create_repo, list_repo_files, snapshot_download
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

def fetch_resume(repo_id, local="/kaggle/working/resume_nllb"):
    try: files = list_repo_files(repo_id, repo_type="model")
    except Exception: print("No repo yet -> fresh."); return None
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(repo_id, allow_patterns="last-checkpoint/*", local_dir=local, repo_type="model")
        print("Resuming."); return os.path.join(local, "last-checkpoint")
    print("No last-checkpoint -> fresh."); return None
create_repo(HUB_CKPT, private=True, exist_ok=True)
resume_ckpt = fetch_resume(HUB_CKPT)

collator = DataCollatorForSeq2Seq(tok, model=model, padding=True)
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEV_BS, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=WARMUP_RATIO,
    fp16=True, optim="adamw_8bit",
    logging_steps=LOG_STEPS, save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    report_to="none", dataloader_num_workers=2, dataloader_pin_memory=False,
    push_to_hub=True, hub_model_id=HUB_CKPT, hub_strategy="checkpoint", hub_private_repo=True)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokd, data_collator=collator)
trainer.train(resume_from_checkpoint=resume_ckpt)

## Save + push (simple — full fine-tune, no adapter to merge)

In [ ]:
trainer.save_model(OUTPUT_DIR)
model.push_to_hub(HUB_MERGED, private=True); tok.push_to_hub(HUB_MERGED)
print("Model pushed to:", HUB_MERGED)

## FLORES-200 scorecard

In [ ]:
from datasets import load_dataset
def load_flores():
    try:
        en=load_dataset("openlanguagedata/flores_plus","eng_Latn",split="devtest")
        sw=load_dataset("openlanguagedata/flores_plus","swh_Latn",split="devtest")
        return [r["text"] for r in en],[r["text"] for r in sw]
    except Exception as e:
        print("flores_plus failed (",e,") -> mirror")
        en=load_dataset("Muennighoff/flores200","eng_Latn",split="devtest")
        sw=load_dataset("Muennighoff/flores200","swh_Latn",split="devtest")
        k="sentence" if "sentence" in en.column_names else en.column_names[0]
        return [r[k] for r in en],[r[k] for r in sw]
src, ref = load_flores(); print("eval sentences:", len(src))

In [ ]:
import torch
model.eval(); model.config.use_cache = True
tgt_id = tok.convert_tokens_to_ids(TGT_LANG)
def translate(sentences, bs=32, max_new=160):
    out=[]
    for i in range(0, len(sentences), bs):
        chunk=sentences[i:i+bs]
        enc=tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            g=model.generate(**enc, forced_bos_token_id=tgt_id, max_new_tokens=max_new, num_beams=5)
        out += tok.batch_decode(g, skip_special_tokens=True)
        if i % (bs*10)==0: print(f"  {i}/{len(sentences)}")
    return out
hyp = translate(src)

In [ ]:
import sacrebleu
chrf=sacrebleu.corpus_chrf(hyp,[ref],word_order=2); bleu=sacrebleu.corpus_bleu(hyp,[ref])
print("="*50)
print("  NLLB 600M en->sw")
print(f"  chrF++ : {chrf.score:5.2f}   (Crane: 56.8)")
print(f"  BLEU   : {bleu.score:5.2f}   (Crane: 27.6)")
print("="*50)
print("BEAT chrF++:", chrf.score>56.8, "| BEAT BLEU:", bleu.score>27.6)